# Cell Surface Visualization

This notebook provides tools to visualize `.ply` surfaces of neurons as rotating 3D objects.

In [ ]:
# Reload modules if changed outside this script
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
from tqdm import tqdm
from IPython.display import display, HTML, Video, Image
from multiprocessing import Pool, cpu_count
import numpy as np
from functools import partial

import sys; sys.path.insert(0, '..')
from scripts.visualize_neurons import create_rotation_mp4_from_h5, create_combined_mp4, create_volume_mp4

In [ ]:
# Path to the cell surfaces
# mesh_directory = Path("../emimesh/results/cells_cube_highres_volfrac/")
mesh_directory = Path("../emimesh/results/cells_lowres/")

In [ ]:
# Print numer of cells
print(f'Number of astrocytes: {len(list(mesh_directory.glob("astrocytes/*")))}')
print(f'Number of microglias: {len(list(mesh_directory.glob("microglias/*")))}')
print(f'Number of neurons: {len(list(mesh_directory.glob("neurons/*")))}')

In [ ]:
mesh_paths = list(mesh_directory.glob("**/mesh.h5"))

print(f"Found {len(mesh_paths)} cell meshes.")
print(list(mesh_paths)[:3])

## Video Generator

This function creates a rotating 3D animation of the mesh surface and saves it as a .mp4

In [ ]:
### Remove all .mp4s

# i = 0
# for mp4 in mesh_directory.rglob("*.mp4"):
#     if "videos" in mp4.parts:
#         continue
#     mp4.unlink()
#     i += 1
# print(f'Removed {i} .mp4 files')

In [ ]:
# Generate MP4 for the first cell
mp4_path = create_rotation_mp4_from_h5((mesh_paths[0], 7), length_sec=20, verbose=True)
display(Video(filename=mp4_path))

In [ ]:
# Generate single png with white background
mp4_path = create_rotation_mp4_from_h5((mesh_paths[0], 7), length_sec=20, verbose=True, single_png=True, bck_white=True)
display(Image(filename=mp4_path))

### Batch Processing

Loop through all neurons in the results folder.

In [ ]:
# mp4
print(f"Using {cpu_count()//2} CPU cores for parallel processing.")
with Pool(processes=cpu_count()//2) as pool:
    tasks = [(path, idx) for idx, path in enumerate(mesh_paths)]
    results = list(tqdm(pool.imap(create_rotation_mp4_from_h5, tasks), total=len(tasks), desc="Generating GIFs", colour="blue"))

print(f"Generated mp4s for {len(results)} neuron surfaces.")

In [ ]:
# png
print(f"Using {cpu_count()//2} CPU cores for parallel processing.")
with Pool(processes=cpu_count()//2) as pool:
    tasks = [(path, idx) for idx, path in enumerate(mesh_paths)]
    func = partial(create_rotation_mp4_from_h5, single_png=True, bck_white=True)
    results = list(tqdm(pool.imap(func, tasks), total=len(tasks), desc="Generating GIFs", colour="blue"))

print(f"Generated mp4s for {len(results)} neuron surfaces.")

## Display all cells

In [ ]:
import base64
start_idx = 16 * 2
max_N = 8

# Collect all mesh.mp4 paths
mp4_paths = [str(p.parent / "mesh.mp4") for p in mesh_paths if (p.parent / "mesh.mp4").exists()]
neuron_labels = {path: p.parent.parent.name for path, p in zip(mp4_paths, mesh_paths)}


html_str = '<div style="display: grid; grid-template-columns: repeat(auto-fill, minmax(500px, 1fr)); gap: 15px; text-align: center;">'
for path in mp4_paths[start_idx:start_idx + max_N]:
    label = neuron_labels[path].replace("_", " ")
    print(label)
    
    # Read .mp4 file and show in HTML
    with open(path, "rb") as f:
        encoded_mp4 = base64.b64encode(f.read()).decode("utf-8")
    
    html_str += '<div style="border: 1px solid #ddd; padding: 5px; border-radius: 5px; background-color: #111;">'
    html_str += f'<video autoplay controls src="data:video/mp4;base64,{encoded_mp4}" style="width: 100%; height: auto;"><br>'
    html_str += f'<strong style="color: white; font-size: 12px;">{label}</strong>'
    html_str += '</div>'

html_str += '</div>'

HTML(html_str)

In [ ]:
import base64

start_idx = 0
max_N = 128

# Collect all mesh.png paths
cell_type = "neuron_5P-ET"
png_paths = [str(p.parent / "mesh.png") for p in mesh_paths if (p.parent / "mesh.png").exists() and cell_type in str(p.parent.parent)]
temp_mesh_paths = [p for p in mesh_paths if (p.parent / "mesh.png").exists() and cell_type in str(p.parent.parent)]
neuron_labels = {path: p.parent.parent.name for path, p in zip(png_paths, temp_mesh_paths)}

html_str = '<div style="display: grid; grid-template-columns: repeat(auto-fill, minmax(500px, 1fr)); gap: 15px; text-align: center;">'
for path in png_paths[start_idx:start_idx + max_N]:
    label = neuron_labels[path].replace("_", " ")

    # Read .png file and show in HTML
    with open(path, "rb") as f:
        encoded_png = base64.b64encode(f.read()).decode("utf-8")

    html_str += '<div style="border: 1px solid #ddd; padding: 5px; border-radius: 5px; background-color: #111;">'
    html_str += f'<img src="data:image/png;base64,{encoded_png}" style="width: 100%; height: auto;" alt="{label}"><br>'
    html_str += f'<strong style="color: white; font-size: 12px;">{label}</strong>'
    html_str += '</div>'

html_str += '</div>'

HTML(html_str)

## Combine N cells into a grid mp4

In [ ]:
# Option to generate a single PNG instead of MP4
single_png = True
suffix = ".png" if single_png else ".mp4"

# for cell_type in ["astrocytes", "microglias", "neurons", "all"]:
for cell_type in ["all"]:
    # Configuration & Downsampling Settings
    start_idx = 0

    # Collect all surface.mp4 paths for the specified cell type
    if cell_type != "all":
        mp4_paths = [p.parent / f"mesh{suffix}" for p in mesh_paths if (p.parent / f"mesh{suffix}").exists() and cell_type in str(p)]
    else:
        mp4_paths = [p.parent / f"mesh{suffix}" for p in mesh_paths if (p.parent / f"mesh{suffix}").exists()] # All

    print(f'{len(mp4_paths)} {suffix} paths found for cell type: {cell_type}')

    # Randomly shuffle the mp4_paths to avoid bias in selection
    np.random.seed(42)  # For reproducibility
    np.random.shuffle(mp4_paths)

    for N in np.arange(2, 20, 4)**2:
    # for N in [15**2]:
        output_mp4_path = create_combined_mp4(
            mp4_paths=mp4_paths,
            output_path=mp4_paths[0].parent.parent.parent.parent / "videos" /  f"{cell_type}_{np.sqrt(N):.0f}x{np.sqrt(N):.0f}.{suffix}",
            N=N,
            start_idx=start_idx,
            verbal=False,
            single_png=single_png
        )

    # Display the combined mp4
    if not single_png:
        display(Video(output_mp4_path, embed=True, width=800, height=600))


# View dense cubes of multiple cells

In [ ]:
path = Path("../emimesh/results/cells_cube_highres_volfrac/branch/size3000_ncells210_0/")
path = Path("../emimesh/results/cells_cube_highres_volfrac/soma/size30000_ncells10_0")
path = Path("../emimesh/results/cells_cube_highres_volfrac/soma/size30000_ncells20_0")
path = Path("../emimesh/results/cells_cube_highres/branch/size10000_ncells550_0")

for cell_type in ["astrocyte", "microglia", "neuron", "all"]:
    gif = create_volume_mp4(
        result_folder=path, 
        primary_opacity=1.0, 
        other_opacity=0.0,
        verbose=True, 
        cell_type=cell_type,
    )

## View surface in Paraview

1. Open Paraview
2. Import all .ply surfaces of interest
3. Apply
4. Select all surfaces -> Filters -> Alphabetical -> Group Datasets -> Apply

## View mesh in Paraview
1. Open Paraview
2. Import .xdmf file
3. Apply
4. Filters -> Alphabetical -> Threshold
5. Increase lower threshold to above 1.0 and apply

